# KMOS Reduction Quick-Look

Inspects the final pipeline products (`_s_i_bkg2d_rv.fits`) produced by
`reduce_kmos.py`.  Three sections:

1. **Inventory** — which OBs / bands were processed and how many exposures each has
2. **Spectra** — aperture-summed 1D spectrum per active IFU, per OB
3. **Spatial maps** — collapsed white-light image per active IFU, per OB

In [ ]:
import glob
import os
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from astropy.io import fits

# ── configuration ─────────────────────────────────────────────────────────────
# Switch BASE to compare sky subtraction methods:
#   "kmos_ob_proc_local"       → zapv2  (kmos_local.cfg)
#   "kmos_ob_proc_local_spark" → spark  (kmos_local_spark.cfg)
BASE        = "kmos_ob_proc_local_spark"  # procdir from the config used
SUFFIX      = "_s_i_bkg2d_rv.fits"        # final pipeline product suffix
APERTURE    = 5                            # aperture box side length in spaxels
EDGE_MARGIN = 4                            # wavelength samples to trim near NaN gaps
# ──────────────────────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlepad": 6,
    "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#555", "axes.linewidth": 0.8,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "#999", "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "legend.framealpha": 0.85, "legend.edgecolor": "0.85",
    "legend.fontsize": 8, "legend.fancybox": True,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})

In [ ]:
# ── helper functions ───────────────────────────────────────────────────────────
import csv
import pathlib

def wavelength_axis(hdr):
    return (np.arange(hdr["NAXIS3"]) + 1 - hdr["CRPIX3"]) * hdr["CDELT3"] + hdr["CRVAL3"]

def active_ifus(hdul):
    return [i for i in range(1, 25)
            if hdul[i].data is not None and np.ndim(hdul[i].data) == 3]

def _erode_valid(finite, margin):
    inv = ~finite
    eroded = inv.copy()
    for s in range(1, margin + 1):
        eroded[s:]  |= inv[:-s]
        eroded[:-s] |= inv[s:]
    return ~eroded

def aperture_spectrum(hdul, ifu, center, box=APERTURE, margin=EDGE_MARGIN):
    cube = np.array(hdul[ifu].data)
    hdr  = hdul[ifu].header
    py, px = center
    h = box // 2
    y0, y1 = max(py - h, 0), min(py + h + 1, cube.shape[1])
    x0, x1 = max(px - h, 0), min(px + h + 1, cube.shape[2])
    patch = cube[:, y0:y1, x0:x1].reshape(cube.shape[0], -1)
    valid = _erode_valid(np.isfinite(patch), margin)
    cleaned = np.where(valid, patch, np.nan)
    rows_ok  = valid.any(axis=1)
    spec = np.full(cube.shape[0], np.nan)
    spec[rows_ok] = np.nansum(cleaned[rows_ok], axis=1)
    return wavelength_axis(hdr), spec

def ifu_center(cube):
    """Geometric centre of the IFU spatial plane (spaxel 7,7 for a 14×14 IFU)."""
    return (cube.shape[1] // 2, cube.shape[2] // 2)

def peak_spaxel(cube, margin=EDGE_MARGIN, min_nbr=0.05):
    valid = _erode_valid(np.isfinite(cube), margin)
    image = np.nansum(np.where(valid, cube, np.nan), axis=0)
    ny, nx = image.shape
    flat = image.ravel()
    order = np.argsort(np.where(np.isfinite(flat), flat, -np.inf))[::-1]
    for idx in order:
        v = flat[idx]
        if not np.isfinite(v) or v <= 0:
            break
        py, px = np.unravel_index(idx, image.shape)
        nbrs = [image[py+dy, px+dx]
                for dy in (-1,0,1) for dx in (-1,0,1)
                if (dy,dx)!=(0,0) and 0<=py+dy<ny and 0<=px+dx<nx]
        nbrs = [n for n in nbrs if np.isfinite(n)]
        if nbrs and np.mean(nbrs) >= min_nbr * v:
            return (py, px)
    return np.unravel_index(np.nanargmax(image), image.shape)

def find_exposures(base=BASE, suffix=SUFFIX):
    result = defaultdict(list)
    for path in sorted(glob.glob(os.path.join(base, "*", "science", f"*{suffix}"))):
        parts = path.replace("\\", "/").split("/")
        ob = parts[-3]
        with fits.open(path) as h:
            band = h[0].header.get("HIERARCH ESO INS GRAT1 ID", "??")
        result[(ob, band)].append(path)
    return dict(result)


# ── emission lines ─────────────────────────────────────────────────────────────
EMISSION_LINES = {
    "[OII]":      0.37270,
    "Hβ":         0.48627,
    "[OIII]4959": 0.49589,
    "[OIII]5007": 0.50082,
    "Hα":         0.65628,
    "[NII]6548":  0.65480,
    "[NII]6583":  0.65835,
    "[SII]6716":  0.67164,
    "[SII]6731":  0.67308,
}
LINE_STYLE = {
    "[OII]":      dict(color="#e377c2", lw=1.8, ls="--"),
    "Hβ":         dict(color="#1f77b4", lw=1.8, ls="-"),
    "[OIII]4959": dict(color="#17becf", lw=1.2, ls=":"),
    "[OIII]5007": dict(color="#17becf", lw=1.8, ls="--"),
    "Hα":         dict(color="#d62728", lw=2.2, ls="-"),
    "[NII]6548":  dict(color="#ff7f0e", lw=1.2, ls=":"),
    "[NII]6583":  dict(color="#ff7f0e", lw=1.8, ls="--"),
    "[SII]6716":  dict(color="#9467bd", lw=1.2, ls=":"),
    "[SII]6731":  dict(color="#9467bd", lw=1.2, ls=":"),
}

def draw_emission_lines(ax, z, wave, compact=False):
    if z is None:
        return
    wlo, whi = np.nanmin(wave), np.nanmax(wave)
    for name, lam_rest in EMISSION_LINES.items():
        lam_obs = lam_rest * (1 + z)
        if not (wlo <= lam_obs <= whi):
            continue
        sty = LINE_STYLE[name]
        c   = sty["color"]

        if compact:
            ax.axvspan(lam_obs - 0.0005, lam_obs + 0.0005,
                       color=c, alpha=0.35, lw=0, zorder=2)
            ax.plot(lam_obs, 1.0, marker="v", markersize=5, color=c,
                    transform=ax.get_xaxis_transform(),
                    clip_on=False, zorder=5, markeredgewidth=0)
            ax.text(lam_obs + 0.0008, 0.97, name,
                    transform=ax.get_xaxis_transform(),
                    rotation=90, ha="left", va="top",
                    fontsize=6, color=c, fontweight="bold",
                    bbox=dict(boxstyle="square,pad=0.05",
                              fc="white", ec="none", alpha=0.6))
        else:
            ax.axvline(lam_obs, alpha=0.75, zorder=2, **sty)
            ax.text(lam_obs, 0.99, name,
                    transform=ax.get_xaxis_transform(),
                    rotation=90, ha="right", va="top",
                    fontsize=8, color=c, fontweight="bold",
                    bbox=dict(boxstyle="square,pad=0.1",
                              fc="white", ec="none", alpha=0.7))


# ── redshift catalogue ─────────────────────────────────────────────────────────
_Z_TABLE   = {}   # (field, arm_target_id) → redshift
_Z_TYPE    = {}   # (field, arm_target_id) → 'spec' | 'photo'
_Z_QUALITY = {}   # (field, arm_target_id) → float quality flag (spec only)

_Z_CSV = pathlib.Path("scripts") / "target_redshifts.csv"
if not _Z_CSV.exists():
    _alt = pathlib.Path.cwd().parent / "scripts" / "target_redshifts.csv"
    if _alt.exists():
        _Z_CSV = _alt

if _Z_CSV.exists():
    with open(_Z_CSV, newline="", encoding="utf-8") as _f:
        for _row in csv.DictReader(_f):
            if _row["redshift"].strip():
                _key = (_row["field"].strip(), _row["arm_target_id"].strip())
                _Z_TABLE[_key]   = float(_row["redshift"].strip())
                _Z_TYPE[_key]    = _row.get("redshift_type", "").strip()
                _q = _row.get("quality", "").strip()
                try:
                    _Z_QUALITY[_key] = float(_q)
                except (ValueError, TypeError):
                    _Z_QUALITY[_key] = np.nan
    _n_spec  = sum(1 for v in _Z_TYPE.values() if v == "spec")
    _n_photo = sum(1 for v in _Z_TYPE.values() if v == "photo")
    print(f"Loaded {len(_Z_TABLE)} redshifts from {_Z_CSV}")
    print(f"  spec-z: {_n_spec}   photo-z: {_n_photo}")
else:
    print(f"WARNING: redshift CSV not found at '{_Z_CSV}'")
    print(f"  cwd = {pathlib.Path.cwd()}")
    print("  Emission lines will not be drawn. "
          "Make sure scripts/target_redshifts.csv exists.")

def get_z(hdul, ifu):
    hdr0  = hdul[0].header
    field = (hdr0.get("ESO OBS TARG NAME") or "").strip()
    key   = f"ESO OCS ARM{ifu} NAME"
    if key in hdr0:
        tid = str(hdr0[key]).strip()
    elif f"HIERARCH {key}" in hdr0:
        tid = str(hdr0[f"HIERARCH {key}"]).strip()
    else:
        return None
    if not field or not tid:
        return None
    return _Z_TABLE.get((field, tid))

def get_z_meta(hdul, ifu):
    """Return (z, z_type, quality) for this IFU."""
    hdr0  = hdul[0].header
    field = (hdr0.get("ESO OBS TARG NAME") or "").strip()
    key   = f"ESO OCS ARM{ifu} NAME"
    if key in hdr0:
        tid = str(hdr0[key]).strip()
    elif f"HIERARCH {key}" in hdr0:
        tid = str(hdr0[f"HIERARCH {key}"]).strip()
    else:
        return None, None, np.nan
    if not field or not tid:
        return None, None, np.nan
    k = (field, tid)
    return _Z_TABLE.get(k), _Z_TYPE.get(k), _Z_QUALITY.get(k, np.nan)


# ── pipeline stage helpers ─────────────────────────────────────────────────────
STAGES = [
    ("",              "reconstructed",    "#888888"),
    ("_s",            "sky-subtracted",   "#1f77b4"),
    ("_s_i",          "+ illumination",   "#ff7f0e"),
    ("_s_i_bkg2d",    "+ background",     "#2ca02c"),
    ("_s_i_bkg2d_rv", "+ rv correction",  "#d62728"),
]

def stage_path(final_path, stage_suffix):
    base = final_path.replace("_s_i_bkg2d_rv.fits", "")
    return base + stage_suffix + ".fits"

## 1 — Inventory

In [ ]:
exposures = find_exposures()

if not exposures:
    print(f"No files matching '*{SUFFIX}' found under '{BASE}'.")
    print("Is the pipeline still running, or does BASE need updating?")
else:
    print(f"{'OB':45s}  {'band':6s}  {'exposures':>9s}")
    print("-" * 65)
    for (ob, band), paths in sorted(exposures.items()):
        print(f"{ob:45s}  {band:6s}  {len(paths):>9d}")
    print("-" * 65)
    print(f"Total: {sum(len(v) for v in exposures.values())} exposures across "
          f"{len(exposures)} (OB, band) combinations")

## 2 — Spectra

One figure per `(OB, band)` combination.  Each subplot = one active IFU.  
Multiple exposures of the same OB are overplotted in light grey; the
exposure-median is drawn on top in colour.

In [ ]:
PLOT_KEYS = sorted(exposures.keys())

for (ob, band) in PLOT_KEYS:
    paths = exposures[(ob, band)]

    ifu_set = set()
    for p in paths:
        with fits.open(p) as h:
            ifu_set.update(active_ifus(h))
    ifu_list = sorted(ifu_set)
    n = len(ifu_list)
    if n == 0:
        print(f"{ob} / {band}: no active IFUs, skipping")
        continue

    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 6.5, nrows * 2.8),
                             constrained_layout=True)
    fig.suptitle(f"{ob}  [{band}]  —  {len(paths)} exposure(s)", y=1.01)
    axes_flat = np.array(axes).ravel()

    for ax_i, ifu in enumerate(ifu_list):
        ax = axes_flat[ax_i]
        spectra, wave, z = [], None, None

        for p in paths:
            with fits.open(p) as h:
                if ifu not in active_ifus(h):
                    continue
                cube = np.array(h[ifu].data)
                ctr  = ifu_center(cube)
                w, s = aperture_spectrum(h, ifu, ctr)
                spectra.append(s)
                if wave is None:
                    wave = w
                if z is None:
                    z = get_z(h, ifu)

        if not spectra or wave is None:
            ax.set_visible(False)
            continue

        for s in spectra:
            ax.plot(wave, s, color="#cccccc", linewidth=0.5, alpha=0.7)
        med = np.nanmedian(np.vstack(spectra), axis=0)
        ax.plot(wave, med, color="tab:blue", linewidth=1.1)

        draw_emission_lines(ax, z, wave, compact=True)

        z_label = f"  z={z:.4f}" if z is not None else ""
        ax.set_title(f"IFU {ifu}{z_label}", pad=4)
        ax.set_xlabel("λ (µm)", fontsize=8)
        ax.set_ylabel("flux", fontsize=8)
        ax.tick_params(labelsize=8)

    for ax in axes_flat[n:]:
        ax.set_visible(False)

    plt.show()

## 3 — Spatial maps

Wavelength-collapsed (summed) white-light image per active IFU, per OB.
The red cross marks the IFU geometric centre used for spectral extraction.

In [ ]:
# one representative exposure per (OB, band) — the first one
REP_EXPOSURE = {key: paths[0] for key, paths in exposures.items()}

for (ob, band), path in sorted(REP_EXPOSURE.items()):
    with fits.open(path) as h:
        ifu_list = active_ifus(h)
        n = len(ifu_list)
        if n == 0:
            continue

        ncols = 6
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.2, nrows * 2.0),
                                 constrained_layout=True)
        fig.suptitle(f"{ob}  [{band}]  —  spatial maps  ({os.path.basename(path)})")
        axes_flat = np.array(axes).ravel()

        for ax_i, ifu in enumerate(ifu_list):
            ax = axes_flat[ax_i]
            cube = np.array(h[ifu].data)
            valid = _erode_valid(np.isfinite(cube), EDGE_MARGIN)
            image = np.nansum(np.where(valid, cube, np.nan), axis=0)
            ctr = ifu_center(cube)

            im = ax.imshow(image, origin="lower", aspect="auto",
                           cmap="viridis",
                           vmin=np.nanpercentile(image, 2),
                           vmax=np.nanpercentile(image, 98))
            ax.plot(ctr[1], ctr[0], "r+", markersize=6, markeredgewidth=1)
            ax.set_title(f"IFU {ifu}", pad=3)
            ax.tick_params(labelsize=6)

        for ax in axes_flat[n:]:
            ax.set_visible(False)

        plt.show()

## 4 — Pipeline stages

One exposure × one IFU, spectrum at every reduction stage overplotted.
The raw reconstructed cube is in grey; each subsequent step in a distinct colour.
Emission lines are marked at the target's redshift (dashed = catalogue, solid = strong).

Set `STAGES_OB`, `STAGES_IFU` to `None` to auto-pick (first OB, first active IFU with a known redshift).

In [ ]:
STAGES_OB  = None   # e.g. "KMOS.2026-01-12T053655.243" — None = auto
STAGES_IFU = None   # e.g. 5 — None = auto
STAGES_HA_HALF = 0.028   # µm half-width for Hα zoom

if not exposures:
    print(f"No exposures found in '{BASE}' — pipeline still running or BASE needs updating.")
else:
    # ── auto-select OB and IFU ─────────────────────────────────────────────────
    _ob_band = sorted(exposures.keys())[0] if STAGES_OB is None else \
               next(k for k in exposures if k[0] == STAGES_OB)
    _ob, _band = _ob_band
    _paths = exposures[_ob_band]
    _rep   = _paths[0]

    with fits.open(_rep) as _h:
        _ifus = active_ifus(_h)
        if STAGES_IFU is not None:
            _ifu = STAGES_IFU
        else:
            _ifu = next((i for i in _ifus if get_z(_h, i) is not None), _ifus[0])
        _ctr = ifu_center(np.array(_h[_ifu].data))
        _z   = get_z(_h, _ifu)

    print(f"OB: {_ob}  band: {_band}  IFU: {_ifu}  z: {_z}")

    # ── load all available stages ──────────────────────────────────────────────
    _stage_data = []
    for suf, label, color in STAGES:
        sp = stage_path(_rep, suf)
        if not os.path.isfile(sp):
            print(f"  stage '{label}' not found, skipping")
            continue
        with fits.open(sp) as h:
            if _ifu not in active_ifus(h):
                print(f"  IFU {_ifu} inactive at stage '{label}', skipping")
                continue
            w, s = aperture_spectrum(h, _ifu, _ctr)
        _stage_data.append((suf, label, color, w, s))

    n_stages = len(_stage_data)

    # ── Figure 1: stacked subplots, one per stage (full wavelength range) ─────
    fig, axes = plt.subplots(n_stages, 1,
                             figsize=(11, 2.6 * n_stages),
                             sharex=True,
                             constrained_layout=True)
    if n_stages == 1:
        axes = [axes]

    fig.suptitle(f"{_ob}  [{_band}]  IFU {_ifu}"
                 + (f"  z = {_z:.4f}" if _z else ""),
                 fontsize=11)

    for row, (suf, label, color, w, s) in enumerate(_stage_data):
        ax = axes[row]

        if row > 0:
            _, _, _, w_prev, s_prev = _stage_data[row - 1]
            ax.plot(w_prev, s_prev, color="#cccccc", linewidth=0.9,
                    alpha=0.8, label=_stage_data[row - 1][1], zorder=1)

        ax.plot(w, s, color=color, linewidth=1.4, label=label, zorder=2)
        draw_emission_lines(ax, _z, w, compact=False)

        ax.set_ylabel("flux", fontsize=8)
        ax.set_title(label, loc="left", pad=3, fontsize=9, color=color)
        ax.legend(loc="upper right", fontsize=7)
        ax.tick_params(labelsize=8)

    axes[-1].set_xlabel("λ (µm)")
    plt.show()

    # ── Figure 2: Hα zoom — all stages overplotted ────────────────────────────
    if _z is not None:
        ha_obs = 0.65628 * (1 + _z)
        wlo, whi = ha_obs - STAGES_HA_HALF, ha_obs + STAGES_HA_HALF

        # check at least the final stage has coverage
        _, _, _, _wf, _sf = _stage_data[-1]
        if _wf.min() <= ha_obs <= _wf.max():
            fig2, ax2 = plt.subplots(figsize=(8, 4), constrained_layout=True)
            fig2.suptitle(
                f"{_ob}  [{_band}]  IFU {_ifu}  z={_z:.4f}"
                f"  —  Hα + [NII] zoom",
                fontsize=10)

            for suf, label, color, w, s in _stage_data:
                mask = (w >= wlo) & (w <= whi)
                ax2.plot(w[mask], s[mask], color=color, linewidth=1.4,
                         label=label, alpha=0.9)

            draw_emission_lines(ax2, _z, _wf[(_wf >= wlo) & (_wf <= whi)],
                                compact=False)

            ax2.set_xlim(wlo, whi)
            ax2.set_xlabel("λ (µm)", fontsize=9)
            ax2.set_ylabel("flux", fontsize=9)
            ax2.legend(fontsize=8, loc="upper right")
            ax2.tick_params(labelsize=8)
            plt.show()
        else:
            print(f"Hα at z={_z:.4f} → {ha_obs:.4f} µm is outside the wavelength range — no zoom plot.")
    else:
        print("No redshift for this IFU — skipping Hα zoom.")

## 5 — Sky method comparison

Final products (`_s_i_bkg2d_rv.fits`) from **zapv2** and **spark** sky subtraction
overlaid on the same axes for every active IFU.  Both methods use the same aperture
centre (derived from the zapv2 cube) so flux differences reflect sky subtraction only.
A thin difference trace (spark − zapv2) is shown offset below each pair for residual inspection.

In [ ]:
COMPARE = {
    "zapv2": "kmos_ob_proc_local",
    "spark": "kmos_ob_proc_local_spark",
}
METHOD_COLOR = {
    "zapv2": "#1f77b4",
    "spark": "#d62728",
}

def _robust_scale(spec):
    """84th–16th percentile half-width as a robust noise estimate."""
    finite = spec[np.isfinite(spec)]
    if finite.size == 0:
        return 1.0
    lo, hi = np.nanpercentile(finite, [16, 84])
    s = (hi - lo) / 2.0
    return s if s > 0 else 1.0

# ── load both runs ────────────────────────────────────────────────────────────
_all_exps = {}
for _method, _base in COMPARE.items():
    _exps = find_exposures(base=_base)
    if _exps:
        _all_exps[_method] = _exps
    else:
        print(f"  {_method} ({_base}): no output yet — skipping")

if len(_all_exps) < 2:
    print("Need both runs to compare. Run the missing pipeline first.")
else:
    _common = sorted(set(_all_exps["zapv2"]) & set(_all_exps["spark"]))
    if not _common:
        print("No (OB, band) combinations in common between both runs.")

    for (ob, band) in _common:
        ifu_sets = {}
        for method in ("zapv2", "spark"):
            s = set()
            for p in _all_exps[method][(ob, band)]:
                with fits.open(p) as h:
                    s.update(active_ifus(h))
            ifu_sets[method] = s
        ifu_list = sorted(ifu_sets["zapv2"] & ifu_sets["spark"])
        n = len(ifu_list)
        if n == 0:
            print(f"{ob} / {band}: no IFUs common to both runs")
            continue

        ncols = 2
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(ncols * 6.5, nrows * 3.2),
                                 constrained_layout=True)
        fig.suptitle(f"{ob}  [{band}]  —  zapv2 (blue) vs spark (red)  [normalised to own RMS]",
                     y=1.01)
        axes_flat = np.array(axes).ravel()

        # build spectra dict once, reuse in Hα zoom cell below
        _cmp_spectra = {}   # (ob, band, ifu) → {method: (wave, spec), z: z}

        for ax_i, ifu in enumerate(ifu_list):
            ax = axes_flat[ax_i]
            spectra = {}
            scales  = {}
            wave_ref = None
            z = None

            ctr = None
            ref_path = _all_exps["zapv2"][(ob, band)][0]
            with fits.open(ref_path) as h:
                if ifu in active_ifus(h):
                    ctr = ifu_center(np.array(h[ifu].data))
                    z = get_z(h, ifu)

            if ctr is None:
                ax.set_visible(False)
                continue

            for method in ("zapv2", "spark"):
                stacks = []
                for p in _all_exps[method][(ob, band)]:
                    with fits.open(p) as h:
                        if ifu not in active_ifus(h):
                            continue
                        w, s = aperture_spectrum(h, ifu, ctr)
                        stacks.append(s)
                        if wave_ref is None:
                            wave_ref = w
                if stacks:
                    raw = np.nanmedian(np.vstack(stacks), axis=0)
                    sc  = _robust_scale(raw)
                    spectra[method] = raw / sc
                    scales[method]  = sc

            if len(spectra) < 2 or wave_ref is None:
                ax.set_visible(False)
                continue

            # store normalised spectra for the Hα zoom cell
            _cmp_spectra[(ob, band, ifu)] = {"wave": wave_ref, "z": z, **spectra}

            for method in ("zapv2", "spark"):
                sc_label = f"{scales[method]:.2e}" if method in scales else "?"
                ax.plot(wave_ref, spectra[method],
                        color=METHOD_COLOR[method], linewidth=1.0,
                        label=f"{method} (÷{sc_label})", alpha=0.9)

            diff = spectra["spark"] - spectra["zapv2"]
            finite_diff = diff[np.isfinite(diff)]
            offset = (np.nanmin([spectra["zapv2"], spectra["spark"]]) -
                      1.8 * (np.nanstd(finite_diff) if finite_diff.size else 1))
            ax.plot(wave_ref, diff + offset,
                    color="#666666", linewidth=0.7, alpha=0.75,
                    label="spark−zapv2 (norm)")
            ax.axhline(offset, color="#bbbbbb", linewidth=0.5, ls="--", zorder=0)

            draw_emission_lines(ax, z, wave_ref, compact=True)

            z_label = f"  z={z:.4f}" if z is not None else ""
            ax.set_title(f"IFU {ifu}{z_label}", pad=4)
            ax.set_xlabel("λ (µm)", fontsize=8)
            ax.set_ylabel("flux / RMS", fontsize=8)
            ax.tick_params(labelsize=8)
            if ax_i == 0:
                ax.legend(fontsize=7, loc="upper right")

        for ax in axes_flat[n:]:
            ax.set_visible(False)

        plt.show()

    print("\nNote: each spectrum is divided by its own noise RMS so zapv2 (ADU) and")
    print("spark (flux-calibrated) are on the same scale. For a unit-matched comparison,")
    print("rerun zapv2 with telluric_in_sof = True in kmos_local.cfg.")

### 5b — Hα + [NII] zoom

Same data as above, x-axis restricted to ±0.028 µm around the redshifted Hα.
IFUs without a known redshift or whose Hα falls outside the band are skipped.
Runs the previous cell first if `_cmp_spectra` is not yet defined.

In [ ]:
HA_HALF = 0.028   # µm half-width: covers Hα + [NII]6548/6583

if "_cmp_spectra" not in dir():
    print("Run the comparison cell above first.")
else:
    # group by (ob, band) preserving order
    from itertools import groupby
    _keys_by_ob = defaultdict(list)
    for (ob, band, ifu) in sorted(_cmp_spectra):
        _keys_by_ob[(ob, band)].append(ifu)

    for (ob, band), ifu_list in _keys_by_ob.items():
        # keep only IFUs with a known z and Hα in range
        _zoom_ifus = []
        for ifu in ifu_list:
            entry = _cmp_spectra[(ob, band, ifu)]
            z = entry["z"]
            wave = entry["wave"]
            if z is None:
                continue
            ha_obs = 0.65628 * (1 + z)
            if (ha_obs - HA_HALF) >= wave.min() and (ha_obs + HA_HALF) <= wave.max():
                _zoom_ifus.append(ifu)

        n = len(_zoom_ifus)
        if n == 0:
            print(f"{ob} / {band}: no IFUs with Hα in range, skipping zoom")
            continue

        ncols = 2
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(ncols * 6.5, nrows * 3.0),
                                 constrained_layout=True)
        fig.suptitle(f"{ob}  [{band}]  —  Hα + [NII] region  |  zapv2 (blue) vs spark (red)",
                     y=1.01)
        axes_flat = np.array(axes).ravel()

        for ax_i, ifu in enumerate(_zoom_ifus):
            ax = axes_flat[ax_i]
            entry = _cmp_spectra[(ob, band, ifu)]
            wave  = entry["wave"]
            z     = entry["z"]
            ha_obs = 0.65628 * (1 + z)
            wlo, whi = ha_obs - HA_HALF, ha_obs + HA_HALF

            for method in ("zapv2", "spark"):
                ax.plot(wave, entry[method],
                        color=METHOD_COLOR[method], linewidth=1.2,
                        label=method, alpha=0.9)

            # difference trace offset below
            diff = entry["spark"] - entry["zapv2"]
            finite_diff = diff[np.isfinite(diff)]
            mask = (wave >= wlo) & (wave <= whi)
            visible = np.concatenate([entry[m][mask] for m in ("zapv2", "spark")])
            finite_v = visible[np.isfinite(visible)]
            y_min = finite_v.min() if finite_v.size else 0
            offset = y_min - 1.8 * (np.nanstd(finite_diff) if finite_diff.size else 1)
            ax.plot(wave, diff + offset,
                    color="#666666", linewidth=0.8, alpha=0.75,
                    label="spark−zapv2")
            ax.axhline(offset, color="#bbbbbb", linewidth=0.5, ls="--", zorder=0)

            draw_emission_lines(ax, z, wave, compact=False)

            ax.set_xlim(wlo, whi)
            if finite_v.size:
                pad = 0.15 * (finite_v.max() - y_min + 1e-6)
                ax.set_ylim(offset - pad, finite_v.max() + pad)

            ax.set_title(f"IFU {ifu}  z={z:.4f}", pad=4)
            ax.set_xlabel("λ (µm)", fontsize=8)
            ax.set_ylabel("flux", fontsize=8)
            ax.tick_params(labelsize=8)
            if ax_i == 0:
                ax.legend(fontsize=7, loc="upper left")

        for ax in axes_flat[n:]:
            ax.set_visible(False)

        plt.show()

## 6 — Stacked rest-frame spectrum

Shift every IFU with a known redshift to the rest frame, resample onto a common
wavelength grid, and median-stack.  A detection in the stack is possible even if
individual IFU S/N is too low.  Both sky methods are shown; the grey band is
the pixel-wise 1σ bootstrap uncertainty on the stack.

In [ ]:
from scipy.interpolate import interp1d

# ── config ────────────────────────────────────────────────────────────────────
REST_LO      = 0.580   # µm
REST_HI      = 0.720   # µm
REST_STEP    = 5e-4    # µm (~1.5× YJ pixel scale)
STACK_Z_TYPE = "spec"  # "spec" | "photo" | "all"
STACK_MIN_Q  = 2.0     # minimum zCOSMOS quality flag (spec-z only; NaN = accept)

rest_wave = np.arange(REST_LO, REST_HI, REST_STEP)

# ── stack builder ─────────────────────────────────────────────────────────────
def build_stack(cmp_spectra, method, z_type=STACK_Z_TYPE, min_q=STACK_MIN_Q):
    rows, labels = [], []
    for (ob, band, ifu), entry in cmp_spectra.items():
        z    = entry.get("z")
        ztyp = entry.get("z_type", "")
        zq   = entry.get("z_quality", np.nan)
        if z is None or method not in entry:
            continue
        if z_type != "all" and ztyp != z_type:
            continue
        if z_type == "spec" and not np.isnan(zq) and zq < min_q:
            continue

        obs_wave  = entry["wave"]
        spec      = entry[method]
        rest_obs  = obs_wave / (1.0 + z)
        sc        = _robust_scale(spec)
        if sc == 0:
            continue
        f = interp1d(rest_obs, spec / sc, kind="linear",
                     bounds_error=False, fill_value=np.nan)
        rows.append(f(rest_wave))
        labels.append(f"IFU {ifu} z={z:.4f} ({ztyp} q={zq})")

    if not rows:
        return rest_wave, None, None, None, []

    mat     = np.vstack(rows)
    n_valid = np.sum(np.isfinite(mat), axis=0)
    med     = np.nanmedian(mat, axis=0)

    rng  = np.random.default_rng(42)
    n_bs = 300
    bs   = np.array([
        np.nanmedian(mat[rng.integers(0, len(rows), len(rows))], axis=0)
        for _ in range(n_bs)
    ])
    err = np.nanstd(bs, axis=0)
    return rest_wave, med, err, n_valid, labels

# ── plot ──────────────────────────────────────────────────────────────────────
if "_cmp_spectra" not in dir() or not _cmp_spectra:
    print("Run Section 5 (sky method comparison) first to populate _cmp_spectra.")
else:
    # attach z_type and quality to _cmp_spectra entries (from _Z_TYPE/_Z_QUALITY)
    # (cell-2 now stores these; Section 5 built _cmp_spectra without them)
    _ref_path = list(find_exposures(base=list(COMPARE.values())[0]).values())[0][0]
    for (ob, band, ifu), entry in _cmp_spectra.items():
        if "z_type" not in entry:
            _zf = list(find_exposures(base=COMPARE["zapv2"]).values())
            _p  = _zf[0][0] if _zf else None
            if _p:
                with fits.open(_p) as _h:
                    _, _zt, _zq = get_z_meta(_h, ifu)
                entry["z_type"]    = _zt or ""
                entry["z_quality"] = _zq

    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
    fig.suptitle(
        f"Rest-frame stacked spectra  ({STACK_Z_TYPE}-z only, quality ≥ {STACK_MIN_Q})",
        fontsize=11)

    for ax, method, color in zip(axes,
                                  ("zapv2", "spark"),
                                  (METHOD_COLOR["zapv2"], METHOD_COLOR["spark"])):
        w, med, err, n_valid, labels = build_stack(_cmp_spectra, method)
        if med is None:
            ax.text(0.5, 0.5, f"No {method} spectra pass filter",
                    transform=ax.transAxes, ha="center")
            continue

        ax.fill_between(w, med - err, med + err,
                        color=color, alpha=0.20, lw=0, label="±1σ (bootstrap)")
        ax.plot(w, med, color=color, lw=1.3, label=f"{method} stack  (N={len(labels)})")
        ax.axhline(0, color="#aaa", lw=0.6, ls="--", zorder=0)

        draw_emission_lines(ax, 0.0, w, compact=False)

        ax.set_ylabel("flux / RMS", fontsize=9)
        ax.set_title(method, loc="left", fontsize=9, color=color)
        ax.text(0.01, 0.97, f"IFUs in stack: {len(labels)}",
                transform=ax.transAxes, va="top", fontsize=8, color="#444")
        ax.legend(loc="upper right", fontsize=8)

    axes[-1].set_xlabel("λ_rest (µm)", fontsize=9)
    axes[-1].set_xlim(REST_LO, REST_HI)
    plt.show()

    # ── per-IFU list ──────────────────────────────────────────────────────────
    print(f"\nIFUs included in stack ({STACK_Z_TYPE}-z, quality ≥ {STACK_MIN_Q}):")
    _, _, _, _, _lbl = build_stack(_cmp_spectra, "zapv2")
    for l in _lbl:
        print(f"  {l}")

    # ── continuum S/N ─────────────────────────────────────────────────────────
    print("\nContinuum S/N (stack median / bootstrap σ, window 0.640–0.650 µm rest):")
    for method in ("zapv2", "spark"):
        w, med, err, _, _ = build_stack(_cmp_spectra, method)
        if med is None:
            continue
        mask = (w >= 0.640) & (w <= 0.650)
        if mask.sum() < 3:
            continue
        snr = np.nanmedian(np.abs(med[mask]) / np.where(err[mask] > 0, err[mask], np.nan))
        print(f"  {method}: {snr:.1f}")